In [ ]:
import json
from datetime import datetime
from pathlib import Path
import numpy as np
from random import shuffle
import random
import collections
import matplotlib.pyplot as plt

---

In [ ]:
# Set time gap (in seconds) to split conversations
TIME_THRESHOLD = 60 * 60 * 8  # 10 minutes

In [ ]:
# Load Telegram JSON export
with open("result.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
messages = data["messages"]

In [ ]:
messages

In [ ]:
# Step 1: Detect your own user ID (you can hardcode this if known)
your_name = "Алексей Семенчук"
your_id = None
for msg in messages:
    if msg.get("from") == your_name:
        your_id = msg.get("from_id")
        break

In [ ]:
# Step 2: Filter and process messages
processed = []
for msg in messages:
    if msg.get("type") != "message":
        continue
    if not msg.get("text"):  # skip empty or media-only messages
        if not msg.get("sticker_emoji"):
            continue
        else:
            text = msg["sticker_emoji"]
    else:
        if isinstance(msg["text"], list):  # sometimes it's list of entities
            text = "".join(t if isinstance(t, str) else t.get("text", "") for t in msg["text"])
        else:
            text = msg["text"]

    processed.append({
        "from": "Леша" if msg["from"] == "Алексей Семенчук" else "Соня",
        "text": text,
        "data": str(msg["date"]),
        # "timestamp": int(msg["date_unixtime"])
    })


In [ ]:
processed

In [ ]:
with open("result_splited.json", "w", encoding="utf-8") as f:
    json.dump(processed, f, ensure_ascii=False, indent=2)

In [ ]:
# Step 3: Split into conversations
conversations = []
current_conv = []
prev_time = None

for msg in processed:
    if prev_time is not None and (msg["timestamp"] - prev_time > TIME_THRESHOLD):
        if current_conv:
            conversations.append(current_conv)
            current_conv = []
    prev_time = msg["timestamp"]
    current_conv.append({"from": msg["from"], "value": msg["text"]})

if current_conv:
    conversations.append(current_conv)

In [ ]:
length = np.array([len(conv) for conv in conversations])
print('Number of conversations:', len(length))
print(f"Mean: {np.mean(length)} +- {np.std(length)}")

In [ ]:
# Step 4: Save as ShareGPT format
output = [{"conversations": conv} for conv in conversations if len(conv) > 1]

output_path = Path("sharegpt_dataset.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved {len(output)} conversations to {output_path}")

---

In [ ]:
with open("result_splited.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
data[0]

In [ ]:
delta_context = 10
delta_response = 10
max_responses = 2

new_data = []
for conv in data:
    for i, message in enumerate(conv['conversations']):
        if message['from'] == 'Соня' and '?' in message['value']:
            new_item = {}

            new_item["instruction"] = message['value']

            context = '\n'.join([f"{m['from']}: {m['value']}" for m in conv['conversations'][max(0, i - delta_context):i]])
            new_item["context"] = context

            response = []
            counter = 0
            for j in range(i+1, min(len(conv['conversations']), i+delta_response)):
                if conv['conversations'][j]['from'] == 'Леша':
                    response.append(conv['conversations'][j]['value'])
                    counter += 1
                if counter >= max_responses:
                    break
            new_item["response"] = '\n'.join(response)

            if response == []:
                continue

            new_data.append(new_item)

In [ ]:
with open('conversation_dialog.json', "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)

---

In [ ]:
with open("dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
new_data = []

for d in data:
    new_data.append({
        'instruction': d['question'],
        'input': "",
        'output': d['answers']
    })

In [ ]:
with open('dataset_alpaca.json', "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)

---

In [ ]:
import random

In [ ]:
with open("result_splited.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
new_data = []

for i, message in enumerate(data):
    if message["from"] == "Соня" and '?' in message["text"]:
        question = message["text"]

        counter = 0
        answers = []
        j = 0
        window = random.randint(1, 5)
        while counter <= window and i + j < len(data):
            if data[i + j]['from'] == 'Леша':
                answers.append(data[i + j]['text'])
                counter += 1
            j += 1

        new_data.append({
            'instruction': question,
            'input': "",
            'output': '\n'.join(answers)
        })


In [ ]:
with open('dataset_alpaca_dump_random.json', "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)

---

In [ ]:
with open("result_splited.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
lesha = 0
sophi = 0
for message in data:
    if message['from'] == 'Леша':
        lesha += 1
    elif message['from'] == 'Соня':
        sophi += 1
    else:
        print('Unknowen user')

print(f'Леша: {lesha}')
print(f'Соня: {sophi}')

In [ ]:
ids = []
for i, message in enumerate(data):
    if message['from'] == 'Леша':
        ids.append(i)

In [ ]:
SIZE = 10_000
step = len(ids) // SIZE
window = 10

new_data = []
for i in range(0, len(ids), step):
    context = [data[id] for id in ids[max(0, i-window):i]]
    answer = data[ids[i]]
    new_data.append({
        "instruction": [f"{message['from']}: {message['text']}" for message in context],
        "input": "",
        "output": answer['text']
    })

In [ ]:
with open('dataset_alpaca_plain.json', "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)

---

In [ ]:
with open("result.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
counter = 0
names = set()
for message in data['messages']:
    if message['type'] == "message":
        names.add(message['from'])
        if message.get('reply_to_message_id', None) is not None and message['from'] == "Соня🤍":
            # print(message)
            counter += 1
print(counter)


In [ ]:
maped_data = dict()

for message in data['messages']:
    if message['type'] == "message":
        id = message['id']
        maped_data[id] = message

In [ ]:
new_data = []

for key, value in maped_data.items():
    if value.get('reply_to_message_id', None) is not None and value['from'] == "Алексей Семенчук":
        if maped_data.get(value['reply_to_message_id'], None) is None:
            continue

        repl_message = maped_data[value['reply_to_message_id']]
        raw_repl_text = repl_message['text']

        # Process the reply text if it's not a simple string
        if isinstance(raw_repl_text, list):
            processed_text_parts = []
            for item in raw_repl_text:
                if isinstance(item, str):
                    processed_text_parts.append(item)
                elif isinstance(item, dict) and 'text' in item:
                    # Append the text content from dictionary objects
                    processed_text_parts.append(item['text'])
                # Add other conditions here if you need to handle other types/structures
            repl_text = "".join(processed_text_parts).strip()
            if not repl_text: # Skip if the processed text is empty
                # print('Processed reply text is empty')
                # print(raw_repl_text)
                continue
        elif isinstance(raw_repl_text, str):
            repl_text = raw_repl_text
        else:
            # Handle or skip other unexpected types
            print(f'Unexpected reply text type: {type(raw_repl_text)}')
            print(raw_repl_text)
            continue

        # Ensure the final repl_text is a non-empty string before proceeding
        if not isinstance(repl_text, str) or not repl_text:
             print('Final reply text is not a valid string or is empty')
             print(f"Original: {raw_repl_text}")
             print(f"Processed: {repl_text}")
             continue

        repl_name = 'Леша' if repl_message['from'] == 'Алексей Семенчук' else 'Соня'

        instruction = f"{repl_name}: {repl_text}"
        output = value['text'] # Assuming output text also might need processing?

        # Potentially process output text similar to repl_text if needed
        if isinstance(output, list):
             processed_output_parts = []
             for item in output:
                 if isinstance(item, str):
                     processed_output_parts.append(item)
                 elif isinstance(item, dict) and 'text' in item:
                     processed_output_parts.append(item['text'])
             output = "".join(processed_output_parts).strip()
        elif not isinstance(output, str):
             print(f"Unexpected output text type: {type(output)}")
             print(output)
             # Decide how to handle non-string output, e.g., skip or convert
             output = str(output) # Example: convert to string representation

        if not output: # Skip if output is empty after processing
             continue

        new_data.append({
            "instruction": instruction,
            "input": "",
            "output": output
        })


In [ ]:
len(new_data)

In [ ]:
shuffle(new_data)
with open('dataset_alpaca_repl.json', "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)

---

In [ ]:
with open("result.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
maped_data = dict()

for message in data['messages']:
    if message['type'] == "message":
        id = message['id']
        maped_data[id] = message

In [ ]:
def get_text(message):
    raw_text = message['text']

    # Process the reply text if it's not a simple string
    if isinstance(raw_text, list):
        processed_text_parts = []
        for item in raw_text:
            if isinstance(item, str):
                processed_text_parts.append(item)
            elif isinstance(item, dict) and 'text' in item:
                # Append the text content from dictionary objects
                processed_text_parts.append(item['text'])
            # Add other conditions here if you need to handle other types/structures
        text = "".join(processed_text_parts).strip()
        if not text: # Skip if the processed text is empty
            print('Processed reply text is empty')
            print(raw_text)
            return None
    elif isinstance(raw_text, str):
        text = raw_text
    else:
        # Handle or skip other unexpected types
        print(f'Unexpected reply text type: {type(raw_text)}')
        print(raw_text)
        return None

    # Ensure the final repl_text is valid
    if not isinstance(text, str):
        print('Final reply text is not a valid string')
        print(f"Original: {raw_text}")
        print(f"Processed: {text}")
        return None

    # Ensure the final repl_text is a non-empty string before proceeding
    if not text:
        return None

    return text


In [ ]:
def delta_time(message_a, message_b):
    return int(message_b['date_unixtime']) - int(message_a['date_unixtime'])

In [ ]:
def get_next_messages(message, maped_data, delta_min=10, delta_max=60 * 10):
    id = message['id']
    name = 'Леша' if message['from'] == 'Алексей Семенчук' else 'Соня'
    assert name == 'Леша', "Соня is not implemented"

    messages = []

    while True:
        next_message = maped_data.get(id + 1, None)
        while not next_message:
            # print(f"Next message not found for {id}, try next")
            id += 1
            next_message = maped_data.get(id + 1, None)

        next_name = 'Леша' if next_message['from'] == 'Алексей Семенчук' else 'Соня'
        if next_name == 'Соня':
            return messages

        if next_message.get('reply_to_message_id', None) is not None:
            return messages

        next_text = get_text(next_message)
        # if next_text is None:
        #     print(f"Next text is None for {id}")
        #     print(f"Next message: {next_message}")
        # elif '?' in next_text:
        #     messages.append(next_text)

        if next_text is not None:
            # if next_text[-1] == '?'
            if delta_time(message, next_message) < delta_min:
                messages.append(next_text)
            elif next_text[-1] == '?':
                if delta_time(message, next_message) < delta_max:
                    messages.append(next_text)
                    # return messages
            # elif '?' in next_text:
            #     messages.append(next_text)


        id += 1


In [ ]:
def get_context(message, maped_data):
    if message.get('reply_to_message_id', None) is None:
        return None, []

    if maped_data.get(message['reply_to_message_id'], None) is None:
        print(f"Reply message not found: {message['reply_to_message_id']}")
        return None, []


    repl_message = maped_data[message['reply_to_message_id']]
    repl_name = 'Леша' if repl_message['from'] == 'Алексей Семенчук' else 'Соня'
    repl_text = get_text(repl_message)
    if repl_text is None:
        return None, []
    repl_output = f"{repl_name}: {repl_text}"

    context_output, context_context = get_context(repl_message, maped_data)

    repl_context = []
    if context_output is not None:
        repl_context.append(context_output)
    if context_context is not None:
        repl_context.extend(context_context)

    return repl_output, repl_context

In [ ]:
new_data = []
context_counter = []
next_messages_counter = []
outputs = []

for key, value in maped_data.items():
    if value['from'] == "Алексей Семенчук":
        input, context = get_context(value, maped_data)
        if input is None:
            continue

        name = 'Леша' if value['from'] == 'Алексей Семенчук' else 'Соня'
        output = get_text(value)
        if output is None:
            continue

        if len(context) > 0:
            context_counter.append(len(context))

        next_messages = get_next_messages(value, maped_data, delta_min=10, delta_max=60 * 10)
        if len(next_messages) > 0:
            next_messages_counter.append(len(next_messages))
            outputs.append((output, next_messages))
            # print(f"Message:")
            # print(f"\t{output}")
            # print(f"\t{next_messages}")
            # print(f"Message + next messages: {output} + {next_messages}")

        new_data.append({
            "context": context[::-1],
            "input": input,
            # "output": (output + '\n' + '\n'.join(next_messages)).strip("\n")
            "output": [output, *next_messages]
        })


In [ ]:
for i, j in random.choices(outputs, k=10):
    print(i)
    print(j)
    print('-' * 100)

In [ ]:
len(new_data)

In [ ]:
len(context_counter)

In [ ]:
collections.Counter(context_counter)

In [ ]:
len(next_messages_counter)

In [ ]:
collections.Counter(next_messages_counter)

In [ ]:
shuffle(new_data)
with open('dataset_context_min_max_long_output.json', "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)

In [ ]:
deltas = list(range(0, 20))
messages_counter = []

for d in deltas:
    next_messages_counter = []
    for key, value in maped_data.items():
        if value['from'] == "Алексей Семенчук":
            input, context = get_context(value, maped_data)
            if input is None:
                continue

            name = 'Леша' if value['from'] == 'Алексей Семенчук' else 'Соня'
            output = get_text(value)
            if output is None:
                continue

            next_messages = get_next_messages(value, maped_data, delta=d)
            if len(next_messages) > 0:
                next_messages_counter.append(len(next_messages))
                # print(f"Message:")
                # print(f"\t{output}")
                # print(f"\t{next_messages}")
                # print(f"Message + next messages: {output} + {next_messages}")
    messages_counter.append(next_messages_counter)


In [ ]:
for d, mess in zip(deltas, messages_counter):
    print(f"{d}: {len(mess)}")


In [ ]:
plt.scatter(deltas, [len(mess) for mess in messages_counter])